# 01 First Look Audit

This notebook performs the initial table-by-table profile pass on the raw Olist source files.
The data is treated as read-only source of truth.

In [2]:
from pathlib import Path
import pandas as pd

base_dir = Path('../data/raw')

raw_tables = {
    'customers': 'olist_customers_dataset.csv',
    'geolocation': 'olist_geolocation_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'order_payments': 'olist_order_payments_dataset.csv',
    'order_reviews': 'olist_order_reviews_dataset.csv',
    'orders': 'olist_orders_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'sellers': 'olist_sellers_dataset.csv',
    'category_name_translation': 'product_category_name_translation.csv',
}

frames = {name: pd.read_csv(base_dir / file_name) for name, file_name in raw_tables.items()}


def show_profile(name: str, df: pd.DataFrame):
    print(f'\\n=== {name.upper()} ===')
    print('shape:', df.shape)
    print('row_count:', len(df))
    print('dtypes:')
    print(df.dtypes)
    print('\\nhead():')
    print(df.head().to_string())
    print('\\ninfo():')
    df.info()
    print('\\nunique key counts:')
    key_columns = [c for c in ['order_id', 'customer_id', 'product_id', 'seller_id'] if c in df.columns]
    print(df[key_columns].nunique(dropna=False))

## customers

Grain: one row per customer_id, representing the customer master record for a given order history.

In [3]:
show_profile('customers', frames['customers'])

\n=== CUSTOMERS ===
shape: (99441, 5)
row_count: 99441
dtypes:
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object
\nhead():
                        customer_id                customer_unique_id  customer_zip_code_prefix          customer_city customer_state
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0                     14409                 franca             SP
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3                      9790  sao bernardo do campo             SP
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e                      1151              sao paulo             SP
3  b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c                      8775        mogi das cruzes             SP
4  4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066    

## geolocation

Grain: one row per geolocation zip code prefix and coordinate pair, used as a lookup reference for city/state geography.

In [4]:
show_profile('geolocation', frames['geolocation'])

\n=== GEOLOCATION ===
shape: (1000163, 5)
row_count: 1000163
dtypes:
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object
\nhead():
   geolocation_zip_code_prefix  geolocation_lat  geolocation_lng geolocation_city geolocation_state
0                         1037       -23.545621       -46.639292        sao paulo                SP
1                         1046       -23.546081       -46.644820        sao paulo                SP
2                         1046       -23.546129       -46.642951        sao paulo                SP
3                         1041       -23.544392       -46.639499        sao paulo                SP
4                         1035       -23.541578       -46.641607        sao paulo                SP
\ninfo():
<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 

## orders

Grain: one row per order_id, representing the lifecycle state and timestamps for a customer order.

In [5]:
show_profile('orders', frames['orders'])

\n=== ORDERS ===
shape: (99441, 8)
row_count: 99441
dtypes:
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object
\nhead():
                           order_id                       customer_id order_status order_purchase_timestamp    order_approved_at order_delivered_carrier_date order_delivered_customer_date order_estimated_delivery_date
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15          2017-10-04 19:55:00           2017-10-10 21:25:13           2017-10-18 00:00:00
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27          2018-07-26 14:31:00           2018-08-07 15:27

## order_items

Grain: one row per order item line, identified by order_id plus order_item_id, representing a product sold within an order.

In [6]:
show_profile('order_items', frames['order_items'])

\n=== ORDER_ITEMS ===
shape: (112650, 7)
row_count: 112650
dtypes:
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object
\nhead():
                           order_id  order_item_id                        product_id                         seller_id  shipping_limit_date   price  freight_value
0  00010242fe8c5a6d1ba2dd792cb16214              1  4244733e06e7ecb4970a6e2683c13e61  48436dade18ac8b2bce089ec2a041202  2017-09-19 09:45:35   58.90          13.29
1  00018f77f2f0320c557190d7a144bdd3              1  e5f2d52b802189ee658865ca93d83a8f  dd7ddc04e1b6c2c614352b383efe2d36  2017-05-03 11:05:13  239.90          19.93
2  000229ec398224ef6ca0657da4fc703e              1  c777355d18b72b67abbeef9df44fd0fd  5b51032eddd242adc84c38acab88f23d  2018-01-18 14:48:30  199.00          17.87
3  00024acbcdf0a6daa1e931b038114c75     

## order_payments

Grain: one row per payment event for an order, with payment_sequential distinguishing multiple payment records on the same order.

In [7]:
show_profile('order_payments', frames['order_payments'])

\n=== ORDER_PAYMENTS ===
shape: (103886, 5)
row_count: 103886
dtypes:
order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64
dtype: object
\nhead():
                           order_id  payment_sequential payment_type  payment_installments  payment_value
0  b81ef226f3fe1789b1e8b2acac839d17                   1  credit_card                     8          99.33
1  a9810da82917af2d9aefd1278f1dcfa0                   1  credit_card                     1          24.39
2  25e8ea4e93396b6fa0d3dd708e76c1bd                   1  credit_card                     1          65.71
3  ba78997921bbcdc1373bb41e913ab953                   1  credit_card                     8         107.78
4  42fdf880ba16b47b59251dd489d4441a                   1  credit_card                     2         128.45
\ninfo():
<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 

## order_reviews

Grain: one row per review_id, representing a customer review record attached to an order.

In [9]:
show_profile('order_reviews', frames['order_reviews'])

\n=== ORDER_REVIEWS ===
shape: (99224, 7)
row_count: 99224
dtypes:
review_id                    str
order_id                     str
review_score               int64
review_comment_title         str
review_comment_message       str
review_creation_date         str
review_answer_timestamp      str
dtype: object
\nhead():
                          review_id                          order_id  review_score review_comment_title                                                                                review_comment_message review_creation_date review_answer_timestamp
0  7bc2406110b926393aa56f80a40eba40  73fc7af87114b39712e6da79b0a377eb             4                  NaN                                                                                                   NaN  2018-01-18 00:00:00     2018-01-18 21:46:59
1  80e641a11e56f04c1ad469d5645fdfde  a548910a1c6147796b98fdf73dbeba33             5                  NaN                                                                      

## products

Grain: one row per product_id, representing the catalog attribute profile of a sold product.

In [10]:
show_profile('products', frames['products'])

\n=== PRODUCTS ===
shape: (32951, 9)
row_count: 32951
dtypes:
product_id                        str
product_category_name             str
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object
\nhead():
                         product_id  product_category_name  product_name_lenght  product_description_lenght  product_photos_qty  product_weight_g  product_length_cm  product_height_cm  product_width_cm
0  1e9e8ef04dbcff4541ed26657ea517e5             perfumaria                 40.0                       287.0                 1.0             225.0               16.0               10.0              14.0
1  3aa071139cb16b67ca9e5dea641aaa2f                  artes                 44.0                       276.0                 1.0            1000.0               30.0    

## sellers

Grain: one row per seller_id, representing the seller master record for marketplace fulfillment.

In [11]:
show_profile('sellers', frames['sellers'])

\n=== SELLERS ===
shape: (3095, 4)
row_count: 3095
dtypes:
seller_id                   str
seller_zip_code_prefix    int64
seller_city                 str
seller_state                str
dtype: object
\nhead():
                          seller_id  seller_zip_code_prefix        seller_city seller_state
0  3442f8959a84dea7ee197c632cb2df15                   13023           campinas           SP
1  d1b65fc7debc3361ea86b5f14c68d2e2                   13844         mogi guacu           SP
2  ce3ad9de960102d0677a81f5d0bb7b2d                   20031     rio de janeiro           RJ
3  c0f3eea2e14555b6faeea3dd58c1b1c3                    4195          sao paulo           SP
4  51a04a8a6bdcb23deccc82b0b80742cf                   12914  braganca paulista           SP
\ninfo():
<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3

## category_name_translation

Grain: one row per product_category_name, mapping the local Portuguese category label to its English translation.

In [12]:
show_profile('category_name_translation', frames['category_name_translation'])

\n=== CATEGORY_NAME_TRANSLATION ===
shape: (71, 2)
row_count: 71
dtypes:
product_category_name            str
product_category_name_english    str
dtype: object
\nhead():
    product_category_name product_category_name_english
0            beleza_saude                 health_beauty
1  informatica_acessorios         computers_accessories
2              automotivo                          auto
3         cama_mesa_banho                bed_bath_table
4        moveis_decoracao               furniture_decor
\ninfo():
<class 'pandas.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   product_category_name          71 non-null     str  
 1   product_category_name_english  71 non-null     str  
dtypes: str(2)
memory usage: 1.2 KB
\nunique key counts:
Series([], dtype: float64)
